# 02 · Compétences, salaires, profils et tendances

Ce notebook réutilise les modules de `src/analysis/` : tu peux donc comparer ses résultats avec ceux des scripts en ligne de commande.

**Prérequis** : avoir lancé `python -m src.analysis.nlp.skills_extractor` (table `job_skills`).

In [5]:
import sys
from pathlib import Path

# Racine du projet : fonctionne que le notebook soit lance depuis notebooks/ ou depuis la racine
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import common

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
print("Projet :", ROOT)

from src.analysis.analyse_salaire import salary_overview, salary_by_skill, salary_by_category
from src.skill_gap.gap_analyzer import (
    DEFAULT_PROFILE, compute_gap, coverage_per_offer, load_profile,
    market_demand, unknown_in_profile,
)
from src.analysis.clustering import build_matrix, choose_k, describe_clusters, fit_clusters
from src.analysis.temporal import monthly_share, skill_trends

jobs, js = common.load_all(only_relevant=True)
print(f"{len(jobs)} offres pertinentes, {len(js)} couples (offre, competence)")

Projet : C:\Users\lenovo\Desktop\Tech job Market intelligent plateforme
4311 offres pertinentes, 2735 couples (offre, competence)


## 1. Compétences les plus demandées

In [ ]:
demand = market_demand(jobs, js)
top = demand.head(15).sort_values("demande_pct")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=top, x="demande_pct", y="competence", hue="categorie", dodge=False, ax=ax)
ax.set_xlabel("% des offres"); ax.set_ylabel("")
ax.legend(title="Categorie", loc="lower right")
plt.show()
demand.head(15)

### Nombre de compétences par offre
Une offre sans aucune compétence détectée = motif manquant dans la taxonomie, ou description trop courte.

In [ ]:
per_offer = js.groupby("id").size().reindex(jobs["id"], fill_value=0)
print(f"Offres sans competence : {(per_offer == 0).mean() * 100:.0f} %   |   moyenne : {per_offer.mean():.1f}")
per_offer.plot.hist(bins=range(0, int(per_offer.max()) + 2), figsize=(7, 3.5), color="#5b8def")
plt.xlabel("Competences par offre"); plt.show()

## 2. Salaires

In [ ]:
ov = salary_overview(jobs)
print(f"Offres avec salaire : {ov['avec_salaire']} ({ov['part_pct']} %)")
if ov["mediane"]:
    print(f"Median : {ov['mediane']:.0f} EUR/mois")
    by_skill = salary_by_skill(jobs, js, min_n=5)
    display(by_skill.head(15))
    display(salary_by_category(jobs, js))
else:
    print("Aucun salaire exploitable.")

In [ ]:
if ov["mediane"] and not by_skill.empty:
    best = by_skill.head(15).sort_values("salaire_median")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(best["competence"], best["salaire_median"], color="#5cb85c")
    ax.axvline(ov["mediane"], color="gray", linestyle="--", label="median global")
    ax.set_xlabel("Salaire median (EUR/mois)"); ax.legend()
    plt.show()

## 3. Skill gap : mon profil face au marché
Modifie `config/my_profile.yaml` puis relance cette cellule.

In [ ]:
profile = load_profile(DEFAULT_PROFILE)
gap = compute_gap(demand, profile)
coverage = coverage_per_offer(js, profile)
unknown = unknown_in_profile(demand, profile)
if unknown:
    print("Jamais detectees dans les offres :", unknown)
print(f"Couverture moyenne des offres : {coverage.mean() * 100:.0f} %")
print(f"Offres ou je connais >= 70 % des competences : {(coverage >= 0.7).mean() * 100:.0f} %")
gap[~gap["possede"]].head(10)

## 4. Profils de postes (clustering)
Chaque offre devient un vecteur 0/1 (compétence présente ou non), puis KMeans les regroupe. Le **score de silhouette** mesure la netteté des groupes (proche de 1 = bien séparés).

In [ ]:
mat = build_matrix(js, min_skills=2, min_freq=5)
print(f"{len(mat)} offres x {mat.shape[1]} competences")
if len(mat) >= 20:
    k, scores = choose_k(mat)
    pd.Series(scores, name="silhouette").plot(marker="o", figsize=(6, 3))
    plt.xlabel("k"); plt.ylabel("silhouette"); plt.show()
    print("k retenu :", k)
else:
    print("Pas assez d'offres pour un clustering fiable.")

In [ ]:
if len(mat) >= 20 and k:
    labels, coords = fit_clusters(mat, k)
    summary = describe_clusters(mat, labels, jobs)
    display(summary)
    plot_df = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1],
                            "profil": [summary.set_index("cluster").loc[c, "label"] for c in labels]})
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=plot_df, x="x", y="y", hue="profil", alpha=0.7, ax=ax)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()

## 5. Tendances dans le temps

In [ ]:
trends = skill_trends(jobs, js, min_n=10)
if trends is None:
    print("Une seule date : relance la collecte regulierement pour voir une evolution.")
else:
    display(trends.head(10))
    display(trends.tail(10).iloc[::-1])

In [ ]:
if trends is not None:
    share = monthly_share(jobs, js)
    follow = demand["competence"].head(5).tolist()
    pivot = share[share["competence"].isin(follow)].pivot(index="periode", columns="competence", values="part_pct")
    pivot.plot(marker="o", figsize=(9, 4))
    plt.ylabel("% des offres de la periode"); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()

## Pour aller plus loin
- Ajouter des motifs dans `config/skills_taxonomy.yaml` pour les compétences « jamais détectées ».
- Comparer les résultats avant/après l'intégration de France Travail.
- Exporter les figures pour le rapport PDF (`generate_report.py`).